In [ ]:
# Install the Azure AI Evaluation SDK with remote capabilities
%pip install azure-ai-evaluation[remote]

# Install additional required packages
%pip install azure-ai-projects azure-identity promptflow-azure python-dotenv pandas

### Import Required Libraries

Now let's import all the libraries we'll use throughout this tutorial.

In [ ]:
import os
import json
from dotenv import load_dotenv
import pandas as pd

%load_ext autoreload
%autoreload 2


load_dotenv()  # Load environment variables from .env file

from rich import console
console = console.Console()

from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_CLIENT_ID = os.getenv("AZURE_CLIENT_ID")

# Example usage:
if False:
    print(f"Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
    print(f"Model Deployment Name: {MODEL_DEPLOYMENT_NAME}")
    print(f"OpenAI Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"OpenAI API Key: {AZURE_OPENAI_API_KEY}")
    print(f"OpenAI API Version: {AZURE_OPENAI_API_VERSION}")
    print(f"AI Resource Name: {AZURE_AI_RESOURCE_NAME}")
    print(f"Client ID: {AZURE_CLIENT_ID}")



# Azure Identity for authentication
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Azure AI Evaluation SDK
from azure.ai.evaluation import (
    AzureOpenAIModelConfiguration,
    evaluate
)

# Built-in evaluators - Quality/RAG
from azure.ai.evaluation import (
    GroundednessEvaluator,
    GroundednessProEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    QAEvaluator
)

# Built-in evaluators - Safety
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    ViolenceEvaluator,
    SelfHarmEvaluator
)

# Built-in evaluators - Agent-specific
from azure.ai.evaluation import (
    IntentResolutionEvaluator,
    TaskAdherenceEvaluator,
    ToolCallAccuracyEvaluator
)

# Azure AI Projects
from azure.ai.projects import AIProjectClient

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
# [START task_adherence_evaluator]
import os
from azure.ai.evaluation import TaskAdherenceEvaluator

model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),  # https://<account_name>.services.ai.azure.com
    "api_key": os.environ.get("AZURE_OPENAI_KEY"),
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
}

task_adherence_evaluator = TaskAdherenceEvaluator(model_config=model_config)

query = [
    {"role": "system", "content": "You are a helpful customer service agent."},
    {"role": "user", "content": [{"type": "text", "text": "What is the status of my order #123?"}]},
]

response = [
    {
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call": {
                    "id": "tool_001",
                    "type": "function",
                    "function": {"name": "get_order", "arguments": {"order_id": "123"}},
                },
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "tool_001",
        "content": [
            {"type": "tool_result", "tool_result": '{ "order": { "id": "123", "status": "shipped" } }'}
        ],
    },
    {"role": "assistant", "content": [{"type": "text", "text": "Your order #123 has been shipped."}]},
]

task_adherence_evaluator(query=query, response=response)
# [END task_adherence_evaluator]

# [START task_completion_evaluator]
import os
from azure.ai.evaluation._evaluators._task_completion import _TaskCompletionEvaluator

model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),  # https://<account_name>.services.ai.azure.com
    "api_key": os.environ.get("AZURE_OPENAI_KEY"),
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
}

task_completion_evaluator = _TaskCompletionEvaluator(model_config=model_config)

query = [
    {"role": "system", "content": "You are a travel booking assistant. Help users find and book flights."},
    {
        "role": "user",
        "content": [{"type": "text", "text": "I need to book a flight from London to Paris for tomorrow"}],
    },
]

response = [
    {
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call": {
                    "id": "search_001",
                    "type": "function",
                    "function": {
                        "name": "search_flights",
                        "arguments": {
                            "origin": "London",
                            "destination": "Paris",
                            "departure_date": "2025-08-13",
                        },
                    },
                },
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "search_001",
        "content": [
            {
                "type": "tool_result",
                "tool_result": '{"flights": [{"flight_id": "BA309", "price": "£89", "departure": "10:30", "arrival": "13:45"}, {"flight_id": "AF1234", "price": "£95", "departure": "14:20", "arrival": "17:35"}]}',
            }
        ],
    },
    {
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "I found 2 flights from London to Paris for tomorrow:\n\n1. BA309 departing 10:30, arriving 13:45 - £89\n2. AF1234 departing 14:20, arriving 17:35 - £95\n\nWould you like me to book one of these flights for you?",
            }
        ],
    },
]

tool_definitions = [
    {
        "name": "search_flights",
        "description": "Search for available flights between two cities.",
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {"type": "string", "description": "Departure city"},
                "destination": {"type": "string", "description": "Arrival city"},
                "departure_date": {"type": "string", "description": "Departure date in YYYY-MM-DD format"},
            },
        },
    }
]

task_completion_evaluator(query=query, response=response, tool_definitions=tool_definitions)
# [END task_completion_evaluator]

# [START indirect_attack_evaluator]
import os
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import IndirectAttackEvaluator

azure_ai_project = os.environ.get(
    "AZURE_AI_PROJECT_URL"
)  # https://{resource_name}.services.ai.azure.com/api/projects/{project_name}
credential = DefaultAzureCredential()

indirect_attack_eval = IndirectAttackEvaluator(azure_ai_project=azure_ai_project, credential=credential)
indirect_attack_eval(
    query="What is the capital of France?",
    response="Paris",
)
# [END indirect_attack_evaluator]

# [START groundedness_pro_evaluator]
import os
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import GroundednessProEvaluator

azure_ai_project = os.environ.get(
    "AZURE_AI_PROJECT_URL"
)  # https://{resource_name}.services.ai.azure.com/api/projects/{project_name}
credential = DefaultAzureCredential()

groundedness_pro_eval = GroundednessProEvaluator(azure_ai_project=azure_ai_project, credential=credential)
groundedness_pro_eval(
    query="What shape has 4 equilateral sides?",
    response="Rhombus",
    context="Rhombus is a shape with 4 equilateral sides.",
)
# [END groundedness_pro_evaluator]

# [START tool_call_accuracy_evaluator]
import os
from azure.ai.evaluation import ToolCallAccuracyEvaluator

model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),  # https://<account_name>.services.ai.azure.com
    "api_key": os.environ.get("AZURE_OPENAI_KEY"),
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
}

tool_call_accuracy_evaluator = ToolCallAccuracyEvaluator(model_config=model_config)
tool_call_accuracy_evaluator(
    query="How is the weather in New York?",
    response="The weather in New York is sunny.",
    tool_calls={
        "type": "tool_call",
        "tool_call": {
            "id": "call_eYtq7fMyHxDWIgeG2s26h0lJ",
            "type": "function",
            "function": {"name": "fetch_weather", "arguments": {"location": "New York"}},
        },
    },
    tool_definitions={
        "id": "fetch_weather",
        "name": "fetch_weather",
        "description": "Fetches the weather information for the specified location.",
        "parameters": {
            "type": "object",
            "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
        },
    },
)
# [END tool_call_accuracy_evaluator]

# [START tool_success_evaluator]
import os
import json
from azure.ai.evaluation._evaluators._tool_success import _ToolSuccessEvaluator

model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.environ.get("AZURE_OPENAI_KEY"),
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
}

tool_success_evaluator = _ToolSuccessEvaluator(model_config=model_config)
tool_success_evaluator(
    response=json.loads(
        """[{"createdAt": "2025-08-16T08:39:47Z", "run_id": "run_id22", "role": "assistant", "content": [{"type": "tool_call", "tool_call_id": "call_id557", "name": "get_date", "arguments": {}}]}, {"createdAt": "2025-08-16T08:39:49Z", "run_id": "run_id22", "tool_call_id": "call_id557", "role": "tool", "content": [{"type": "tool_result", "tool_result": {"date_and_time": "2019-09-07 23:59:59"}}]}, {"createdAt": "2025-08-16T08:39:51Z", "run_id": "run_id22", "role": "assistant", "content": [{"type": "tool_call", "tool_call_id": "call_Run1", "name": "get_spending_by_day", "arguments": {"start_date": "2019-10-01", "end_date": "2019-10-31"}}]}, {"createdAt": "2025-08-16T08:39:53Z", "run_id": "run_id22", "tool_call_id": "call_Run1", "role": "tool", "content": [{"type": "tool_result", "tool_result": {"spending": {}}}]}, {"createdAt": "2025-08-16T08:39:54Z", "run_id": "run_id22", "role": "assistant", "content": [{"type": "text", "text": "There are no spending records for October."}]}]"""
    ),
    tool_definitions=json.loads(
        """[{"name": "get_categories", "type": "function", "description": "Retrieve of a spending line id from your spending records."}]"""
    ),
)

tool_success_evaluator(
    response="the agent called get_categories and the call returned the value Electronics",
    tool_definitions="We have a tool named get_categories that takes the spending line id as a input and outputs the category of this spending line",
)
# [END tool_success_evaluator]

# [START tool_output_utilization]
import os
from azure.ai.evaluation import _ToolOutputUtilizationEvaluator

model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.environ.get("AZURE_OPENAI_KEY"),
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
}

tool_output_utilization_evaluator = _ToolOutputUtilizationEvaluator(model_config=model_config)
query = [
    {"role": "system", "content": "You are a customer service assistant helping with order inquiries."},
    {"role": "user", "content": [{"type": "text", "text": "What's the status of order #12345?"}]},
]

response = [
    {
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "call_456",
                "name": "get_order_status",
                "arguments": {"order_id": "12345"},
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "call_456",
        "content": [
            {
                "type": "tool_result",
                "tool_result": {
                    "order_id": "12345",
                    "status": "shipped",
                    "tracking_number": "1Z999AA1234567890",
                    "estimated_delivery": "2024-10-03",
                },
            }
        ],
    },
    {
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Your order #12345 has been shipped! The tracking number is 1Z999AA1234567890 and it's estimated to arrive on October 3rd, 2024.",
            }
        ],
    },
]

tool_definitions = [
    {
        "name": "get_order_status",
        "type": "function",
        "description": "Retrieves current status and details for an order",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}},
    }
]

tool_output_utilization_evaluator(query=query, response=response, tool_definitions=tool_definitions)
# [END tool_output_utilization]

# [START task_navigation_efficiency_evaluator]
from azure.ai.evaluation._evaluators._task_navigation_efficiency import (
    _TaskNavigationEfficiencyEvaluator,
    _TaskNavigationEfficiencyMatchingMode,
)

task_navigation_efficiency_evaluator = _TaskNavigationEfficiencyEvaluator(
    matching_mode=_TaskNavigationEfficiencyMatchingMode.EXACT_MATCH
)

response = [
    {
        "role": "assistant",
        "content": [{"type": "tool_call", "tool_call_id": "call_1", "name": "search", "arguments": {}}],
    },
    {
        "role": "assistant",
        "content": [{"type": "tool_call", "tool_call_id": "call_2", "name": "analyze", "arguments": {}}],
    },
    {
        "role": "assistant",
        "content": [{"type": "tool_call", "tool_call_id": "call_3", "name": "report", "arguments": {}}],
    },
]
ground_truth = ["search", "analyze", "report"]

task_navigation_efficiency_evaluator(response=response, ground_truth=ground_truth)

# Also supports tuple format with parameters for exact parameter matching
response_with_params = [
    {
        "role": "assistant",
        "content": [
            {"type": "tool_call", "tool_call_id": "call_1", "name": "search", "arguments": {"query": "test"}}
        ],
    },
]
ground_truth_with_params = (["search"], {"search": {"query": "test"}})

task_navigation_efficiency_evaluator(response=response_with_params, ground_truth=ground_truth_with_params)
# [END task_navigation_efficiency_evaluator]